# StrikeDetection — fase 3 manual

La partición 9/2/2 está aprobada. Este notebook coordina extracción reanudable y entrenamiento de la cabeza. **El usuario inicia las ejecuciones largas**. Ejecutar todas las celdas con sus valores por defecto solo verifica contratos: los controles `RUN_EXTRACTION` y `RUN_TRAINING` están desactivados.

Guía completa: [STRIKE-ML-MANUAL.md](../../docs/STRIKE-ML-MANUAL.md). Los experimentos iniciales permanecen en `strike-detection-x3d-s.ipynb`.

In [1]:
from pathlib import Path
import json, sys, subprocess
import yaml
import pandas as pd
from IPython.display import display
from ufc_tracker.strike.cache import context, cache_path, extract
from ufc_tracker.strike.training import settings_check, train
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/data/strike_split_v1.json').is_file())
assert Path(sys.prefix).name.lower() == 'strikevision_ml', sys.executable
settings = yaml.safe_load((ROOT / 'configs/models/strike_training_v1.yaml').read_text())
settings_check(settings)
ctx = context(ROOT)
print('Entorno:', sys.executable)
print('Partición:', ctx['split']['status'])
print('Cache:', cache_path(ROOT, ctx))
display(pd.DataFrame(ctx['summary']).T)


Entorno: C:\Users\Cristian\miniconda3\envs\strikevision_ML\python.exe
Partición: approved_by_user
Cache: C:\Users\Cristian\Documents\UFC TRACKER\data\processed\features\strike_x3d_s\c4f34ae24890b96c00c7a1363536d84add9ac3d4c0aafcc95779a0ffa719c011


,fights,windows,labels,categories,fps
train,"[adesanya_pereira_1, fiziev_bahamondes, fiziev...",25592,"{'no_strike': 18411, 'strike': 7181}","[aggressive_men, low_kicks_men, normal_men, pa...","[24.0, 29.97002997002997, 30.0, 59.94004231195..."
validation,"[aldo_font, zhang_joanna_1]",7396,"{'no_strike': 4933, 'strike': 2463}","[aggressive_women, normal_men]",[29.97002997002997]
test,"[holloway_gaethje, joanna_andrade]",7492,"{'no_strike': 5190, 'strike': 2302}","[aggressive_men, low_kicks_women]",[29.97002997002997]


## Extracción de train y validación

Cambia a `True` **solo cuando decidas iniciar** la extracción. Proyección del benchmark: aproximadamente 2 h 15 min, más overhead de integridad/cache, sin garantía.

Interrumpir la celda conserva bloques confirmados de 128 ventanas. Para reanudar, vuelve a ejecutar esta celda. El test no está habilitado. No abrir otra ejecución simultánea en terminal.

In [2]:
RUN_EXTRACTION = False
if RUN_EXTRACTION:
    result = extract(ROOT, ctx, settings, ['train', 'validation'])
    print('Cache terminado:', result)
else:
    print('Extracción desactivada. No se han procesado videos.')


Extracción desactivada. No se han procesado videos.


## Entrenamiento manual de la cabeza

Activar solo después de completar ambos caches. Compara pérdida estándar y pesos de clase de train; el extractor permanece congelado. Selecciona checkpoint y umbral con validación. No evalúa test.

Para reanudar usa el mismo `RUN_NAME`. Si cambias receta, código o datos, usa un nombre diferente. El estado se guarda después de cada época; se repite una época interrumpida.

In [3]:
RUN_NAME = 'head-v1'
RUN_TRAINING = False
if RUN_TRAINING:
    run_dir = train(ROOT, ctx, settings, RUN_NAME)
    print('Entrenamiento terminado:', run_dir)
else:
    print('Entrenamiento desactivado. No se han actualizado pesos.')


Entrenamiento desactivado. No se han actualizado pesos.


## Recarga en un proceso limpio y revisión de validación

Esta celda solo verifica un candidato ya existente; nunca inicia entrenamiento. La prueba es de la cabeza sobre embeddings. La exportación RGB completa, el test reservado y Gradio vienen después de revisar los resultados.

In [4]:
run_dir = ROOT / 'outputs/strike-detection' / RUN_NAME
if (run_dir / 'READY.json').exists():
    result = subprocess.run([sys.executable, '-s', str(ROOT / 'scripts/ml/verify_strike_candidate.py'), '--run-dir', str(run_dir)], capture_output=True, text=True, check=True)
    print(result.stdout)
    report = json.loads((run_dir / 'validation.json').read_text())
    display(pd.DataFrame({k: report['validation'][k] for k in ('model', 'always_no_strike', 'always_strike')}))
    for loss_name in settings['training']['losses']:
        print(loss_name)
        display(pd.read_json(run_dir / loss_name / 'history.json'))
else:
    print('Todavía no hay un candidato completo. Ejecuta manualmente los pasos anteriores cuando lo decidas.')


Todavía no hay un candidato completo. Ejecuta manualmente los pasos anteriores cuando lo decidas.


## Después de ejecutar

Comparte `validation.json`, `reload_check.json` y los historiales. Las probabilidades describen ventanas, no certeza calibrada ni golpes individuales. No se promueve producción automáticamente.

No descongelaremos bloques de X3D ni cambiaremos hardware/coste sin consultarte. La separación aprobada mantiene categorías femeninas diferentes y solo 29,97 FPS en validación/test; esas limitaciones acompañarán el informe.